# 임베딩 모델 파인튜닝

- 임베딩 모델 파인튜닝은 사전 학습된 임베딩 모델을 특정 도메인이나 작업에 맞게 최적화하는 과정입니다.


## 1. 임베딩 모델의 학습 원리

- 의미가 비슷한 문장 쌍에는 높은 임베딩 유사도를, 의미가 다른 문장 쌍에는 낮은 유사도를 반환하도록 임베딩 벡터를 업데이트 하는 방식입니다.
- 임베딩 모델을 학습할 때는 의미가 유사한 문장 쌍과 유사하지 않은 문장 쌍을 대조하여 학습하는 방식, 즉 대조 학습을 활용합니다.


### 1.1 대조학습

- 포지티브 샘플
  - 의미적으로 관련이 있는 문장 쌍을 의미합니다.
  - 예: (기준 문서: "서울의 인구는?", 비교 문서: "서울의 인구는 약 970만명 입니다.")
- 네커티브 샘플
  - 기준 문서는 동일하지만, 비교 문서는 의미적으로 관련이 없거나 관련성이 낮은 문장을 준비하여 이들을 쌍으로 구성한 데이터입니다.
  - 예: (기준 문서: "서울의 인구는?", 비교 문서: "파리는 프랑스의 수도입니다.")


- 포지티브 샘플은 RAG를 수행할 때 사용자가 입력할 만한 검색어를 기준문서, 검색 결과로 유사도가 높게 나오기를 바라는 문서를 관련 있는 문서로 삼아 구성합니다.
- 네거티브 샘플은 RAG상황에서 같은 앵커에 대한 검색 결과에 포함되지 않기를 바라는 문서를 짝지어 구성합니다.


- 포지티브 샘플과 네거티브 샘플 구성
  - 기준 문서를 중심으로 유사도가 높은 쌍인 포지티브 샘플(관련 있는 쌍)과 유사도가 낮은 네거티브 샘플(관련 없는 쌍)을 모두 학습 데이터로 준비합니다.
- 대조 학습
  - 모델이 포지티브 샘플 쌍의 임베딩 간 거리는 가깝게, 네거티브 샘플 쌍의 임베딩 간 거리는 멀게 만들도록 학습합니다.
- 손실 함수 최적화
  - 임베딩 간 유사도를 계산하여, 포지티브 쌍의 임베딩 유사도는 높이고 네거티브 쌍의 임베딩 유사도는 낮추는 방향으로 손실 함수를 최적화합니다.


- 손실 함수는 모델이 예측한 결과와 실제 정답 간의 오차를 계산해 학습을 조정하는 기준이 됩니다.
- MultipleNegativesRankingLoss라는 손실 함수를 사용할 예정입니다.


### 1.2 데이터셋 구성

- 대조 학습에서는 하나의 기준 문서에 대해 하나의 포지티브 샘플과 하나 이상의 네거티브 샘플을 명시적으로 준비해야 합니다.
- 기준 문서는 앵커라고 부릅니다.
- 네거티브 샘플은 포지티브 샘플보다 양이 많을수록 좋습니다.


- 트리플렛 구성
  - 전통적인 방식은 각 학습 데이터를 (앵커, 포지티브, 네거티브) 형태의 트리플렛으로 구성하는 것입니다.


In [ ]:
# 전통적인 트리플렛 구성 예
triplets = [
    # (앵커, 포지티브, 네거티브)
    ("강아지를 기르는 방법", "반려견 양육 가이드", "고양이 사료 추천"),
    ("파이썬 코딩 튜토리얼", "파이썬 프로그래밍 기초", "자바스크립트 입문 강의"),
    # 수천, 수만 개의 트래플렛 필요
]

- 다중 네거티브 구성
  - 실제 모델 학습에서는 하나의 앵커에 여러개의 네거티브 샘플을 포함하는 구성이 더 효과적인 경우가 많습니다.


In [ ]:
# 다중 네거티브 샘플 구성 예
training_data = [
    {
        "anchor": "머신러닝이란?",
        "positive": "기계학습은 데이터로부터 패턴을 찾는 AI 기술입니다.",
        "negatives": [
            "오늘 날씨가 좋네요",
            "내일 회의는 2시에 시작합니다.이 식당의 불고기가 맛있습니다.",
            # 여러 개의 네거티브 샘플
        ],
    },
    # 수천개의 이러한 구조
]

- 다중 네거티브 구성은 학습 효과를 높일 수 있지만, 그만큼 데이터 준비의 난이도도 높아집니다.
- 임베딩 모델을 효과적으로 파인튜닝하기 위해서는, 특히 네거티브 샘플 선정이 가장 까다로운 작업중 하나입니다.


- 네거티브 샘플의 문서는 각 앵커와 관련 없는 텍스트여야만 합니다.
- 적절한 난이도의 네거티브 샘플을 선택해야 합니다.
  - 두 개의 쌍이 너무 관련이 없다면 임베딩 모델이 판단하기 너무 쉬워서 학습 효과가 거의 없게 됩니다.
  - 사람이 보아도 관련이 있는 것인지 관련이 없는 것인지 헷갈릴 정도의 문서 쌍이라면 난이도가 너무 높아져 학습에 오히려 방해가 됩니다.
- 다중 네거티브 샘플을 구성할 경우, 네거티브 샘플을 포지티브 샘플 대비 몇 배로 구성하느냐에 따라 데이터셋 크기가 기하급수적으로 증가하고, 만들어야 하는 데이터의 양이 많아지게 됩니다.


- 임베딩 파인튜닝에서 양질의 네거티브 샘플을 구성하는 것은 종종 전체 학습과정에서 가장 어려운 부분 중 하나입니다.


### 1.3 배치 내 네거티브 샘플링

- 배치 내에서 네거티브 샘플을 선정하는 학습 방법을 사용합니다.
- 이 방법은 명시적인 네거티브 샘플을 별도로 준비할 필요가 없다는 큰 장점이 있습니다.
- 학습 데이터에서 다른 앵커에서 사용하고 있는 샘플을 참고하여 자동으로 네거티브로 활용합니다.
- 이 원리를 이해하려면 배치라는 개념을 알아야 합니다.
- AI모델은 데이터를 적당한 개수의 묶음으로 나누어 학습합니다.
- 예를 들어 데이터가 5000개 이고 배치 크기를 40으로 설정했다면, 데이터를 40개씩 묶어 125회에 걸쳐 학습하게 됩니다.
- 배치란 모델이 한 번에 학습하는 데이터의 단위를 뜻하며, 병렬적으로 데이터를 몇개씩 학습할 것이냐를 의미합니다.


- 배치 내 네거티브 생성 방법은 포지티브 샘플만으로 데이터를 구성하더라도 배치내에서 네거티브 샘플들을 자동으로 만드는 학습 방법입니다.
- 사용자는 학습을 위해 포지티브 샘플만 제공하면 되며, 네거티브 샘플은 학습 시 배치내에서 자동으로 생성됩니다.


- 예를 들어 배치 크기가 4인 경우, 한번의 학습에 네 개의 서로 다른 앵커 문서와 그에 대응하는 포지티브 샘플이 사용되며, 이들 간 교차로 네거티브 샘플 역할도 동시에 수행됩니다.


```
배치 = [
    (앵커문서1, 문서1),
    (앵커문서2, 문서2),
    (앵커문서3, 문서3),
    (앵커문서4, 문서4)
]
```


- 앵커 문서1의 포지티브는 문서1, 나머지는 네거티브로 간주됩니다.
- 앵커 문서2의 포지티브는 문서2, 나머지는 네거티브로 간주됩니다.


- 하나의 배치 안에서 다른 쌍의 문서를 네거티브로 자동 활용하면서 대조 학습을 수행합니다.


```
[
    ("AI란 무엇인가?", "AI는 인간의 지능을 모방한 기술입니다."),
    ("딥러닝이란?", "신경망을 여러 층 쌓아 데이터로부터 학습하는 기계학습 방법입니다."),
    ("Python은 어디에 쓰이나요?", "Python은 데이터 분석, 웹 개발, AI 등에 널리 사용됩니다."),
    ("자연어 처리란?" , "컴퓨터가 인간의 언어를 이해하고 처리하는 AI의 한 분야입니다.")
]
```


### MultipleNegativesRankingLoss

- 이 손실 함수는 포지티브 샘플과의 유사도는 높이고, 네거티브 샘플과는 유사도는 낮추도록 설계되어 있습니다.


In [1]:
from sentence_transformers import SentenceTransformer, losses, InputExample
from torch.utils.data import DataLoader
import torch

# 모델 로드
model = SentenceTransformer("BAAI/bge-m3")

# 훈련 데이터 준비
train_examples = [
    InputExample(texts=["AI란 무엇인가?", "AI는 인간의 지능을 모방한 기술입니다."]),
    InputExample(
        texts=[
            "딥러닝이란?",
            "신경망을 여러 층 쌓아 데이터로부터 학습하는 기계학습 방법입니다.",
        ]
    ),
    InputExample(
        texts=[
            "Python은 어디에 쓰이나요?",
            "Python은 데이터 분석, 웹 개발, AI등에 널리 사용됩니다.",
        ]
    ),
    InputExample(
        texts=[
            "자연어 처리란?",
            "컴퓨터가 인간의 언어를 이해하고 처리하는 AI의 한 분야입니다.",
        ]
    ),
]

# 배치 크기가 클수록 성능이 향상될 수 있지만 GPU에 따라서 최대 비치 크기가 제한됨
batch_size = 32
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=batch_size)

# MultipleNegativesRankingLoss 설정
# 온도(temperature) 파라미터를 조정하여 손실 함수의 강도 조절 가능
loss = losses.MultipleNegativesRankingLoss(
    model, scale=20.0
)  # scale은 temperature의 역수

# 학습 설정
train_loss = losses.MultipleNegativesRankingLoss(model)
warmup_steps = int(len(train_dataloader) * 0.1)  # 전체 훈련 데이터의 10%

# 모델 학습
model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=3,
    warmup_steps=warmup_steps,
    optimizer_params={"lr": 2e-5},
    output_path="./korean-sentence-embedding-model",
)

c:\workspace\python\rag_master\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\workspace\python\rag_master\.venv\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss


- 먼저 sentence_transformers 라이브러리를 통해 BAAI/bge-m3 모델을 기본 모델로 로드합니다.
- 이어 훈련데이터를 InputExample객체의 리스트로 준비합니다.
- 모델은 이러한 문장의 쌍들을 통해 유사한 문장들이 임베딩 공간에서 가깝게 위치하도록 학습합니다.
- DataLoader를 활용하여 배치크기 32로 데이터를 효율적으로 처리하도록 설정합니다.
  - 데이터가 4개밖에 없지만, 실제 상황에서는 데이터가 32개보다 많다고 가정합니다.
- 손실함수로 MultipleNegativesRankingLoss를 채택했습니다. 의미적으로 유사한 문장들은 가깝게, 그렇지 않은 문장들은 멀리 위치시키도록 모델을 유도합니다.
  - scale=20.0 파라미터는 온도의 역수로 손실함숨의 강도를 적절히 조절하는 역할을 합니다.
- 학습과정에서는 워밍업단계를 전체 훈련데이터의 10%로 설정합니다.
- 업데이트 하는 정도를 조절하는 학습률은 2e-5 로 지정합니다.
- model.fit() 함수를 호출하여 모델을 학습합니다.
- 학습횟수를 의미하는 에포크의 경우 3
- 완성된 모델은 korean-sentence-embedding-model 디렉터리에 저장합니다.
